# 01 — Bronze: clientsSame contract as `00_bronze_bids`: ingest as-is, validate the column set,defer every judgement to Silver.The client export carries two quirks worth noting but *not* fixing here — a`2999-12-31` sentinel marking open-ended contracts, and literal `'null'`strings where a value is absent. Both survive into Bronze untouched.

In [0]:
CATALOG = "bronze"SCHEMA = "bid"VOLUME_PATH = "/Volumes/raw/bid/bids/clients.xlsx"TABLE = f"{CATALOG}.{SCHEMA}.clients"spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

In [0]:
from pyspark.sql import functions as Fdf_raw = (    spark.read.format("com.crealytics.spark.excel")    .option("header", "true")    .option("inferSchema", "false")    .option("dataAddress", "'Clients'!A1")    .load(VOLUME_PATH))df_bronze = (    df_raw    .withColumn("_ingested_at", F.current_timestamp())    .withColumn("_source_file", F.lit(VOLUME_PATH)))(    df_bronze.write.format("delta").mode("overwrite")    .option("mergeSchema", "true").saveAsTable(TABLE))print(f"rows: {spark.table(TABLE).count()}")

In [0]:
EXPECTED = {    "client_id", "contract_name", "status", "start_date", "end_date",    "state", "city", "segment", "account_executive", "director",    "manager", "coordinator",}actual = set(df_bronze.columns) - {"_ingested_at", "_source_file"}missing = EXPECTED - actualif missing:    raise ValueError(f"Columns missing from source: {sorted(missing)}")print("Schema check passed.")